# 05 — Explainability (SHAP Analysis)

Generate SHAP explanations for the best model.
Global feature importance, beeswarm plots, and individual patient waterfall plots.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from pathlib import Path

shap.initjs()

## 1. Load Best Model & Data

In [ ]:
from src.model import load_model
from src.explainer import get_shap_explainer, compute_shap_values, get_feature_importance
from src.preprocessing import RISK_LABEL_MAP

processed = Path('../data/processed')
outputs = Path('../outputs')

# Load data
X_test_d = pd.read_csv(processed / 'diabetes_X_test.csv')
y_test_d = pd.read_csv(processed / 'diabetes_y_test.csv').squeeze()
X_test_h = pd.read_csv(processed / 'heart_X_test.csv')
y_test_h = pd.read_csv(processed / 'heart_y_test.csv').squeeze()

# Load best model (XGBoost typically)
model_d = load_model('xgboost', 'diabetes')
model_h = load_model('xgboost', 'heart')
print('Models loaded successfully')

## 2. SHAP — Diabetes

In [ ]:
explainer_d = get_shap_explainer(model_d)
shap_values_d = compute_shap_values(explainer_d, X_test_d)
print(f'SHAP values shape: {shap_values_d.shape}')

### 2.1 Global Feature Importance

In [ ]:
importance_d = get_feature_importance(shap_values_d)
print(importance_d.to_string(index=False))

# Bar plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_d['Feature'], importance_d['Importance'], color='#4e79a7')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Diabetes — Global Feature Importance (SHAP)')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(outputs / 'diabetes_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Beeswarm Plot

Shows how each feature value pushes predictions. Red = high feature value, Blue = low.

In [ ]:
# For multi-class, show SHAP for the High-risk class (class 2)
if len(shap_values_d.shape) == 3:
    shap_high = shap.Explanation(
        values=shap_values_d.values[:, :, 2],
        base_values=shap_values_d.base_values[:, 2],
        data=shap_values_d.data,
        feature_names=shap_values_d.feature_names,
    )
else:
    shap_high = shap_values_d

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_high, show=False)
plt.title('Diabetes — SHAP Beeswarm (High Risk Class)')
plt.tight_layout()
plt.savefig(outputs / 'diabetes_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Individual Patient Explanations

Waterfall plots showing WHY a specific patient received their risk score.

In [ ]:
# Find one patient from each risk class
preds_d = model_d.predict(X_test_d)

for risk_class in [0, 1, 2]:
    idx = np.where(preds_d == risk_class)[0]
    if len(idx) > 0:
        patient_idx = idx[0]
        risk_name = RISK_LABEL_MAP[risk_class]
        print(f'\n=== Patient #{patient_idx} — {risk_name} Risk ===')
        
        if len(shap_values_d.shape) == 3:
            patient_shap = shap.Explanation(
                values=shap_values_d.values[patient_idx, :, risk_class],
                base_values=shap_values_d.base_values[patient_idx, risk_class],
                data=shap_values_d.data[patient_idx],
                feature_names=shap_values_d.feature_names,
            )
        else:
            patient_shap = shap_values_d[patient_idx]
        
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(patient_shap, show=False)
        plt.title(f'Patient #{patient_idx} — {risk_name} Risk')
        plt.tight_layout()
        plt.savefig(outputs / f'diabetes_waterfall_{risk_name.lower()}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 3. SHAP — Heart Disease

In [ ]:
explainer_h = get_shap_explainer(model_h)
shap_values_h = compute_shap_values(explainer_h, X_test_h)

importance_h = get_feature_importance(shap_values_h)
print(importance_h.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_h['Feature'], importance_h['Importance'], color='#e15759')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Heart Disease — Global Feature Importance (SHAP)')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(outputs / 'heart_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.1 Beeswarm — Heart Disease

In [ ]:
if len(shap_values_h.shape) == 3:
    shap_high_h = shap.Explanation(
        values=shap_values_h.values[:, :, 2],
        base_values=shap_values_h.base_values[:, 2],
        data=shap_values_h.data,
        feature_names=shap_values_h.feature_names,
    )
else:
    shap_high_h = shap_values_h

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_high_h, show=False)
plt.title('Heart Disease — SHAP Beeswarm (High Risk Class)')
plt.tight_layout()
plt.savefig(outputs / 'heart_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clinical Alert Examples

In [ ]:
from src.explainer import generate_clinical_alerts, format_prediction_summary

# Pick a high-risk diabetes patient
high_idx = np.where(preds_d == 2)[0]
if len(high_idx) > 0:
    patient_idx = high_idx[0]
    patient_data = X_test_d.iloc[patient_idx].to_dict()
    probabilities = model_d.predict_proba(X_test_d.iloc[[patient_idx]])[0]
    
    if len(shap_values_d.shape) == 3:
        sv = shap_values_d.values[patient_idx, :, 2]
    else:
        sv = shap_values_d.values[patient_idx]
    
    alerts = generate_clinical_alerts(
        patient_data=patient_data,
        shap_values_patient=sv,
        feature_names=list(X_test_d.columns),
        dataset_type='diabetes',
    )
    
    summary = format_prediction_summary(2, probabilities, alerts)
    print(summary)

## 5. Summary

All SHAP plots and clinical alerts have been generated and saved to `outputs/`.

Key findings:
- ...
- ...
- ...